# Supply Chain Service Level, Inventory Risk & Working Capital Intelligence System

Executive portfolio notebook for the complete analytical stack, including policy simulation, stress testing, supplier-lane diagnostics, and quality gates.


In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for p in [cwd, *cwd.parents]:
    if (p / 'data').exists() and (p / 'src').exists() and (p / 'outputs').exists():
        PROJECT_ROOT = p
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Project root not found')

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS = PROJECT_ROOT / 'outputs' / 'reports'
TABLES = PROJECT_ROOT / 'outputs' / 'tables'
print('Project root:', PROJECT_ROOT)


## 1) Core KPI Snapshot

In [ ]:
overall = pd.read_csv(REPORTS / 'kpi_overall_service_health.csv').iloc[0]
impact = pd.read_csv(TABLES / 'impact_overall_summary.csv')
impact_map = dict(zip(impact['metric'], impact['value']))

pd.DataFrame([
    ('Fill rate', f"{overall['overall_fill_rate']:.2%}"),
    ('Stockout rate', f"{overall['overall_stockout_rate']:.2%}"),
    ('Lost sales (observed)', f"EUR {overall['total_lost_sales_revenue']:,.0f}"),
    ('Trapped WC proxy (observed)', f"EUR {impact_map['trapped_working_capital_proxy_observed']:,.0f}"),
    ('12M opportunity proxy (estimated)', f"EUR {impact_map['opportunity_total_12m_proxy']:,.0f}"),
], columns=['Metric', 'Value'])


## 2) Service and Inventory Concentration

In [ ]:
pd.read_csv(REPORTS / 'kpi_service_by_warehouse.csv').sort_values('warehouse_service_risk_proxy', ascending=False).head(8)


In [ ]:
pd.read_csv(REPORTS / 'kpi_excess_and_slow_moving_exposure.csv').sort_values('excess_inventory_value_proxy', ascending=False).head(8)


## 3) Policy Simulation Frontier (Upgrade)

In [ ]:
frontier = pd.read_csv(TABLES / 'policy_simulation_frontier.csv')
frontier


## 4) Monte Carlo Stress Test (Upgrade)

In [ ]:
stress = pd.read_csv(TABLES / 'stress_monte_carlo_lane_results.csv')
stress.head(15)[[
    'supplier_id','warehouse_id','category','product_id',
    'prob_stockout','expected_service_level','severe_event_probability','stress_risk_score','lost_sales_revenue'
]]


## 5) Supplier-Lane Diagnostics (Upgrade)

In [ ]:
lane = pd.read_csv(TABLES / 'supplier_lane_diagnostics.csv')
lane.head(15)[[
    'supplier_id','warehouse_id','category','supplier_lane_risk_score','downstream_stockout_rate','lost_sales_revenue','supplier_service_risk_proxy'
]]


## 6) Quality Gates (Upgrade)

In [ ]:
pre = pd.read_csv(TABLES / 'validation_pre_delivery_checks.csv')
sql = pd.read_csv(TABLES / 'ci_sql_validation_checks.csv')

pd.DataFrame([
    ('Pre-delivery checks', len(pre), int((pre.status=='PASS').sum()), int((pre.status=='FAIL').sum()), int((pre.status=='WARN').sum())),
    ('SQL quality checks', len(sql), int((sql.status=='PASS').sum()), int((sql.status!='PASS').sum()), 0),
], columns=['Gate', 'Checks', 'Pass', 'Fail', 'Warn'])


## 7) Delivery Artifacts

In [ ]:
artifacts = [
    PROJECT_ROOT / 'outputs' / 'dashboard' / 'index.html',
    PROJECT_ROOT / 'outputs' / 'reports' / 'executive_summary.md',
    PROJECT_ROOT / 'outputs' / 'reports' / 'policy_simulation_summary.md',
    PROJECT_ROOT / 'outputs' / 'reports' / 'monte_carlo_stress_summary.md',
    PROJECT_ROOT / 'outputs' / 'reports' / 'supplier_lane_diagnostics_summary.md',
    PROJECT_ROOT / '.github' / 'workflows' / 'analytics-ci.yml',
]

pd.DataFrame([(str(p.relative_to(PROJECT_ROOT)), p.exists()) for p in artifacts], columns=['Artifact', 'Exists'])
